# 03 Upselling, efficiency, and store segmentation

This notebook looks at which stores are strong beyond raw volume. I separate volume from efficiency so that a smaller store with good upselling performance is not automatically ignored.

The segmentation is descriptive. It is meant to support business review, not to claim one store type causes better performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1) Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# 2) Set file paths

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Practicum/Demo File")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
VISUAL_DIR = BASE_DIR / "visuals"

VISUAL_DIR.mkdir(parents=True, exist_ok=True)

store_performance_path = PROCESSED_DIR / "store_performance_summary.csv"
monthly_analysis_path = PROCESSED_DIR / "monthly_analysis_dataset.csv"

print("Store performance file exists:", store_performance_path.exists())
print("Monthly analysis file exists:", monthly_analysis_path.exists())

In [ ]:
# 3) Load data from previous notebook

store_performance = pd.read_csv(store_performance_path)
monthly_analysis = pd.read_csv(monthly_analysis_path)

monthly_analysis["Month"] = pd.to_datetime(monthly_analysis["Month"])

print("Store performance shape:", store_performance.shape)
print("Monthly analysis shape:", monthly_analysis.shape)

display(store_performance.head())
display(monthly_analysis.head())

In [ ]:
# 4) Quick sanity check

# I am checking these fields because the whole notebook depends on them
# If any of these are missing, it means the previous notebook did not save correctly

needed_cols = [
    "Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size",
    "Total_Activation", "PPD", "Boxes", "Hours",
    "Account_Gross", "Accessory_Profit", "Acc_Dollar",
    "PPD_per_Hour", "Accessory_Revenue_per_PPD",
    "Accessory_Revenue_per_Box", "Accessory_Profit_per_Activation",
    "Account_Gross_per_Activation"
]

missing_cols = [col for col in needed_cols if col not in store_performance.columns]

if len(missing_cols) == 0:
    print("All required columns are available.")
else:
    print("Missing columns:", missing_cols)

In [ ]:
# 5) Select main upselling and efficiency metrics

efficiency_cols = [
    "Store_ID",
    "Store_Name",
    "Market_Zone",
    "Store_Type",
    "Store_Size",
    "Total_Activation",
    "PPD",
    "Boxes",
    "Account_Gross",
    "Accessory_Profit",
    "Acc_Dollar",
    "PPD_per_Hour",
    "Accessory_Revenue_per_PPD",
    "Accessory_Revenue_per_Box",
    "Accessory_Profit_per_Activation",
    "Account_Gross_per_Activation"
]

efficiency_df = store_performance[efficiency_cols].copy()

display(efficiency_df.head())

In [ ]:
# 6) Rank stores by accessory profit per activation

# This helps identify stores that generate more accessory profit
# relative to their activation activity.

accessory_profit_rank = (
    efficiency_df
    .sort_values("Accessory_Profit_per_Activation", ascending=False)
    [
        [
            "Store_ID",
            "Store_Name",
            "Total_Activation",
            "Accessory_Profit",
            "Accessory_Profit_per_Activation"
        ]
    ]
)

display(accessory_profit_rank.head(10))

In [ ]:
# 7) Plot accessory profit per activation

plot_df = accessory_profit_rank.sort_values("Accessory_Profit_per_Activation", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["Accessory_Profit_per_Activation"])

plt.title("Accessory Profit per Activation by Store")
plt.xlabel("Accessory Profit per Activation")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "accessory_profit_per_activation_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 8) Rank stores by accessory revenue per box

# This shows how much accessory revenue is generated per device sold.
# It is useful for checking accessory attachment behavior

acc_box_rank = (
    efficiency_df
    .sort_values("Accessory_Revenue_per_Box", ascending=False)
    [
        [
            "Store_ID",
            "Store_Name",
            "Boxes",
            "Acc_Dollar",
            "Accessory_Revenue_per_Box"
        ]
    ]
)

display(acc_box_rank.head(10))

In [ ]:
# 9) Plot accessory revenue per box

plot_df = acc_box_rank.sort_values("Accessory_Revenue_per_Box", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["Accessory_Revenue_per_Box"])

plt.title("Accessory Revenue per Box by Store")
plt.xlabel("Accessory Revenue per Box")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "accessory_revenue_per_box_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 10) Rank stores by PPD per hour

# This is an operational productivity metric.
# It shows how much activity stores generate relative to operating hours.

ppd_hour_rank = (
    efficiency_df
    .sort_values("PPD_per_Hour", ascending=False)
    [
        [
            "Store_ID",
            "Store_Name",
            "PPD",
            "Hours" if "Hours" in efficiency_df.columns else "PPD_per_Hour",
            "PPD_per_Hour"
        ]
    ]
)

display(ppd_hour_rank.head(10))

In [ ]:
# 10b. Rank stores by PPD per hour - safer version

ppd_hour_rank = (
    store_performance
    .sort_values("PPD_per_Hour", ascending=False)
    [
        [
            "Store_ID",
            "Store_Name",
            "PPD",
            "Hours",
            "PPD_per_Hour"
        ]
    ]
)

display(ppd_hour_rank.head(10))

In [ ]:
# 11) Plot PPD per hour

plot_df = ppd_hour_rank.sort_values("PPD_per_Hour", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["PPD_per_Hour"])

plt.title("PPD per Hour by Store")
plt.xlabel("PPD per Hour")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "ppd_per_hour_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 12) Create volume and efficiency scores

# I am using ranks instead of raw values because the metrics have different units.
# Example: Account Gross is dollars, PPD is count and PPD per Hour is a rate.

classification_df = store_performance.copy()

volume_metrics = [
    "Total_Activation",
    "PPD",
    "Account_Gross"
]

efficiency_metrics = [
    "PPD_per_Hour",
    "Accessory_Revenue_per_PPD",
    "Accessory_Revenue_per_Box",
    "Accessory_Profit_per_Activation",
    "Account_Gross_per_Activation"
]

# Higher value is better for all selected metrics.
for col in volume_metrics:
    classification_df[col + "_Rank"] = classification_df[col].rank(ascending=False, method="min")

for col in efficiency_metrics:
    classification_df[col + "_Rank"] = classification_df[col].rank(ascending=False, method="min")

classification_df["Avg_Volume_Rank"] = classification_df[[col + "_Rank" for col in volume_metrics]].mean(axis=1)
classification_df["Avg_Efficiency_Rank"] = classification_df[[col + "_Rank" for col in efficiency_metrics]].mean(axis=1)

n_stores = classification_df["Store_ID"].nunique()

classification_df["Volume_Score"] = 1 - ((classification_df["Avg_Volume_Rank"] - 1) / (n_stores - 1))
classification_df["Efficiency_Score"] = 1 - ((classification_df["Avg_Efficiency_Rank"] - 1) / (n_stores - 1))

display(
    classification_df[
        [
            "Store_ID",
            "Store_Name",
            "Volume_Score",
            "Efficiency_Score",
            "Avg_Volume_Rank",
            "Avg_Efficiency_Rank"
        ]
    ].sort_values("Volume_Score", ascending=False)
)

In [ ]:
# 13) Decide threshold using median score

# Since this is a relative store comparison, I am using the median as the cutoff.
# This means "high" and "low" are based on this group of demo stores only.

volume_cutoff = classification_df["Volume_Score"].median()
efficiency_cutoff = classification_df["Efficiency_Score"].median()

print("Volume cutoff:", round(volume_cutoff, 3))
print("Efficiency cutoff:", round(efficiency_cutoff, 3))

In [ ]:
# 14) Classify stores into four performance groups

def classify_store(row):
    if row["Volume_Score"] >= volume_cutoff and row["Efficiency_Score"] >= efficiency_cutoff:
        return "High Volume + High Efficiency"
    elif row["Volume_Score"] >= volume_cutoff and row["Efficiency_Score"] < efficiency_cutoff:
        return "High Volume + Lower Efficiency"
    elif row["Volume_Score"] < volume_cutoff and row["Efficiency_Score"] >= efficiency_cutoff:
        return "Lower Volume + High Efficiency"
    else:
        return "Lower Volume + Lower Efficiency"

classification_df["Performance_Category"] = classification_df.apply(classify_store, axis=1)

store_classification = classification_df[
    [
        "Store_ID",
        "Store_Name",
        "Market_Zone",
        "Store_Type",
        "Store_Size",
        "Total_Activation",
        "PPD",
        "Account_Gross",
        "Accessory_Profit",
        "PPD_per_Hour",
        "Accessory_Revenue_per_PPD",
        "Accessory_Revenue_per_Box",
        "Accessory_Profit_per_Activation",
        "Account_Gross_per_Activation",
        "Volume_Score",
        "Efficiency_Score",
        "Performance_Category"
    ]
].copy()

display(store_classification.sort_values(["Performance_Category", "Volume_Score"], ascending=[True, False]))

In [ ]:
# 15) Category counts

category_counts = (
    store_classification["Performance_Category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["Performance_Category", "Store_Count"]

display(category_counts)

In [ ]:
# 16) Plot volume vs efficiency classification

plt.figure(figsize=(9, 7))

categories = store_classification["Performance_Category"].unique()

for category in categories:
    temp = store_classification[store_classification["Performance_Category"] == category]
    plt.scatter(
        temp["Volume_Score"],
        temp["Efficiency_Score"],
        label=category,
        s=80,
        alpha=0.8
    )

# Median cutoff lines
plt.axvline(volume_cutoff, linestyle="--")
plt.axhline(efficiency_cutoff, linestyle="--")

for _, row in store_classification.iterrows():
    plt.text(
        row["Volume_Score"] + 0.01,
        row["Efficiency_Score"] + 0.01,
        row["Store_ID"],
        fontsize=8
    )

plt.title("Store Classification: Volume Score vs Efficiency Score")
plt.xlabel("Volume Score")
plt.ylabel("Efficiency Score")
plt.legend(loc="best", fontsize=8)
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "store_classification_volume_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 17) Store type segment analysis

# This is a demo-only field, but it shows how store attributes can be used
# for segmentation if the business has this type of data available.

store_type_segment = (
    store_performance
    .groupby("Store_Type", as_index=False)
    .agg(
        Store_Count=("Store_ID", "nunique"),
        Total_Activation=("Total_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Avg_PPD_per_Hour=("PPD_per_Hour", "mean"),
        Avg_Accessory_Profit_per_Activation=("Accessory_Profit_per_Activation", "mean"),
        Avg_Account_Gross_per_Activation=("Account_Gross_per_Activation", "mean")
    )
    .sort_values("Total_Activation", ascending=False)
)

display(store_type_segment)

In [ ]:
# 18) Plot activation by store type

plot_df = store_type_segment.sort_values("Total_Activation", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["Store_Type"], plot_df["Total_Activation"])

plt.title("Total Activation by Store Type")
plt.xlabel("Total Activation")
plt.ylabel("Store Type")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "segment_total_activation_by_store_type.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 19) Market zone segment analysis

zone_segment = (
    store_performance
    .groupby("Market_Zone", as_index=False)
    .agg(
        Store_Count=("Store_ID", "nunique"),
        Total_Activation=("Total_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Avg_PPD_per_Hour=("PPD_per_Hour", "mean"),
        Avg_Accessory_Profit_per_Activation=("Accessory_Profit_per_Activation", "mean"),
        Avg_Account_Gross_per_Activation=("Account_Gross_per_Activation", "mean")
    )
    .sort_values("Total_Activation", ascending=False)
)

display(zone_segment)

In [ ]:
# 20) Plot activation by market zone

plot_df = zone_segment.sort_values("Total_Activation", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["Market_Zone"], plot_df["Total_Activation"])

plt.title("Total Activation by Market Zone")
plt.xlabel("Total Activation")
plt.ylabel("Market Zone")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "segment_total_activation_by_market_zone.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 21) Store size segment analysis

size_segment = (
    store_performance
    .groupby("Store_Size", as_index=False)
    .agg(
        Store_Count=("Store_ID", "nunique"),
        Total_Activation=("Total_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Avg_PPD_per_Hour=("PPD_per_Hour", "mean"),
        Avg_Accessory_Profit_per_Activation=("Accessory_Profit_per_Activation", "mean"),
        Avg_Account_Gross_per_Activation=("Account_Gross_per_Activation", "mean")
    )
    .sort_values("Total_Activation", ascending=False)
)

display(size_segment)

In [ ]:
# 22) Plot PPD per hour by store size

plot_df = size_segment.sort_values("Avg_PPD_per_Hour", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df["Store_Size"], plot_df["Avg_PPD_per_Hour"])

plt.title("Average PPD per Hour by Store Size")
plt.xlabel("Average PPD per Hour")
plt.ylabel("Store Size")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "segment_ppd_per_hour_by_store_size.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 23) Promotion day analysis

# This uses monthly counts of promotion days.
# It is not causal, but it can show whether months with more promotion days
# tend to have higher activity.

promotion_month_summary = (
    monthly_analysis
    .assign(Promotion_Group=np.where(monthly_analysis["Promotion_Days"] > 0, "Promotion Month", "No Promotion Month"))
    .groupby("Promotion_Group", as_index=False)
    .agg(
        Month_Store_Count=("Store_ID", "count"),
        Avg_Total_Activation=("Total_Activation", "mean"),
        Avg_Account_Gross=("Account_Gross", "mean"),
        Avg_Accessory_Profit=("Accessory_Profit", "mean"),
        Avg_PPD=("PPD", "mean")
    )
)

display(promotion_month_summary)

In [ ]:
# 24) Plot promotion month comparison

plt.figure(figsize=(8, 5))
plt.bar(promotion_month_summary["Promotion_Group"], promotion_month_summary["Avg_Total_Activation"])

plt.title("Average Monthly Activation: Promotion vs No Promotion Months")
plt.xlabel("Promotion Group")
plt.ylabel("Average Monthly Activation")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "promotion_month_activation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 25) Inventory issue analysis

# Here we are checking whether store-months with inventory issues look different
# from normal store-months.

inventory_month_summary = (
    monthly_analysis
    .assign(Inventory_Group=np.where(monthly_analysis["Inventory_Issue_Days"] > 0, "Inventory Issue Month", "No Inventory Issue Month"))
    .groupby("Inventory_Group", as_index=False)
    .agg(
        Month_Store_Count=("Store_ID", "count"),
        Avg_Total_Activation=("Total_Activation", "mean"),
        Avg_Account_Gross=("Account_Gross", "mean"),
        Avg_Accessory_Profit=("Accessory_Profit", "mean"),
        Avg_PPD=("PPD", "mean")
    )
)

display(inventory_month_summary)

In [ ]:
# 26) Plot inventory month comparison

plt.figure(figsize=(8, 5))
plt.bar(inventory_month_summary["Inventory_Group"], inventory_month_summary["Avg_Total_Activation"])

plt.title("Average Monthly Activation: Inventory Issue vs No Issue Months")
plt.xlabel("Inventory Group")
plt.ylabel("Average Monthly Activation")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "inventory_month_activation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 27) Create simple upselling opportunity diagnostic

# This table is useful for the dashboard.
# It gives each store a short business-readable interpretation.

market_avg_acc_profit_per_activation = store_performance["Accessory_Profit_per_Activation"].mean()
market_avg_acc_rev_per_box = store_performance["Accessory_Revenue_per_Box"].mean()
market_avg_ppd_per_hour = store_performance["PPD_per_Hour"].mean()

def store_strength(row):
    strengths = []

    if row["Accessory_Profit_per_Activation"] >= market_avg_acc_profit_per_activation:
        strengths.append("strong accessory profit efficiency")

    if row["Accessory_Revenue_per_Box"] >= market_avg_acc_rev_per_box:
        strengths.append("strong accessory revenue per device")

    if row["PPD_per_Hour"] >= market_avg_ppd_per_hour:
        strengths.append("strong activity productivity")

    if len(strengths) == 0:
        return "No major strength above market average"

    return "; ".join(strengths)


def store_opportunity(row):
    opportunities = []

    if row["Accessory_Profit_per_Activation"] < market_avg_acc_profit_per_activation:
        opportunities.append("improve accessory profit per activation")

    if row["Accessory_Revenue_per_Box"] < market_avg_acc_rev_per_box:
        opportunities.append("improve accessory revenue per device")

    if row["PPD_per_Hour"] < market_avg_ppd_per_hour:
        opportunities.append("improve activity productivity")

    if len(opportunities) == 0:
        return "Maintain current performance and monitor trend"

    return "; ".join(opportunities)


opportunity_diagnostic = store_classification.copy()

opportunity_diagnostic["Store_Strength"] = opportunity_diagnostic.apply(store_strength, axis=1)
opportunity_diagnostic["Store_Opportunity"] = opportunity_diagnostic.apply(store_opportunity, axis=1)

opportunity_diagnostic = opportunity_diagnostic[
    [
        "Store_ID",
        "Store_Name",
        "Market_Zone",
        "Store_Type",
        "Store_Size",
        "Performance_Category",
        "Store_Strength",
        "Store_Opportunity"
    ]
]

display(opportunity_diagnostic.sort_values(["Performance_Category", "Store_ID"]))

In [ ]:
# saving this output so the next notebook / Power BI can reuse the same table
# 28) Save segmentation and efficiency outputs

store_classification_path = PROCESSED_DIR / "store_classification.csv"
opportunity_diagnostic_path = PROCESSED_DIR / "upselling_opportunity_diagnostic.csv"
store_type_segment_path = PROCESSED_DIR / "segment_store_type_summary.csv"
zone_segment_path = PROCESSED_DIR / "segment_market_zone_summary.csv"
size_segment_path = PROCESSED_DIR / "segment_store_size_summary.csv"
promotion_month_summary_path = PROCESSED_DIR / "promotion_month_summary.csv"
inventory_month_summary_path = PROCESSED_DIR / "inventory_month_summary.csv"

store_classification.to_csv(store_classification_path, index=False)
opportunity_diagnostic.to_csv(opportunity_diagnostic_path, index=False)
store_type_segment.to_csv(store_type_segment_path, index=False)
zone_segment.to_csv(zone_segment_path, index=False)
size_segment.to_csv(size_segment_path, index=False)
promotion_month_summary.to_csv(promotion_month_summary_path, index=False)
inventory_month_summary.to_csv(inventory_month_summary_path, index=False)

print("Saved store classification:", store_classification_path)
print("Saved upselling opportunity diagnostic:", opportunity_diagnostic_path)
print("Saved segment summaries.")

In [ ]:
# 29) Short notebook summary

print("Upselling Efficiency and Segmentation Summary")
print("--------------------------------------------")
print("Stores analyzed:", store_classification["Store_ID"].nunique())
print("Volume cutoff:", round(volume_cutoff, 3))
print("Efficiency cutoff:", round(efficiency_cutoff, 3))

print("\nStore categories:")
display(category_counts)

print("\nTop 5 stores by accessory profit per activation:")
display(accessory_profit_rank.head(5))

print("\nTop 5 stores by PPD per hour:")
display(ppd_hour_rank.head(5))

print("\nOpportunity diagnostic preview:")
display(opportunity_diagnostic.head(10))